# Notebook 01: Prompt Foundations

## Learning objectives

By the end of this notebook, you will be able to:

1. Write a clear task instruction.
2. Provide relevant context.
3. Specify the intended audience.
4. Define constraints.
5. Specify and evaluate an output format.

**Keywords:** Task • Context • Audience • Constraints • Output Format

## Our experiment

We will begin with a vague prompt and improve it one component at a time.

We will use the same model throughout so that we can observe the effect of
changing the prompt.

In [2]:
!ollama list

NAME                       ID              SIZE      MODIFIED     
nomic-embed-text:latest    0a109f422b47    274 MB    14 hours ago    
llama3.2:1b                baf6a787fdff    1.3 GB    16 hours ago    
qwen2.5:1.5b               65ec06548149    986 MB    2 weeks ago     


In [ ]:
## 1. Setup

Before running this notebook:

1. Install and start Ollama.
2. Open a terminal and download the model:

    ollama pull qwen3:8b 
    # Use Ollama pull qwen2.5:3B FOR THOSE WHO HAS LESS THAN 8 GB RAM

3. Check your installed models:

    ollama list

Run Jupyter on the same computer as Ollama.

If you already have another model installed, enter its exact name in the
configuration cell below.

Run notebook cells from top to bottom using Shift + Enter.

In [4]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import time
import requests
import pandas as pd

from IPython.display import display, Markdown

OLLAMA_URL = "http://localhost:11434"

# Change this to your installed model name.
# MODEL = "qwen3:8b"
MODEL = "qwen2.5:1.5b"

# Keep these settings fixed throughout this notebook.
TEMPERATURE = 0.2
MAX_GENERATED_TOKENS = 4096
REQUEST_TIMEOUT = 600

print("Selected model:", MODEL)

Selected model: qwen2.5:1.5b


In [8]:
# Check that Ollama is running and the model is installed.

try:
    response = requests.get(
        f"{OLLAMA_URL}/api/tags",
        timeout=10
    )
    response.raise_for_status()

except requests.RequestException as error:
    raise RuntimeError(
        "Cannot connect to Ollama. Start Ollama and check OLLAMA_URL. "
        "Jupyter must be able to reach the Ollama server."
    ) from error

available_models = [
    item["name"]
    for item in response.json().get("models", [])
]

print("Installed models:")
for name in available_models:
    print("-", name)

if MODEL not in available_models:
    raise RuntimeError(
        f"{MODEL} is not installed. "
        f"Run 'ollama pull {MODEL}' in a terminal, "
        "or change MODEL to an installed model name."
    )

print("\nReady to begin.")

Installed models:
- nomic-embed-text:latest
- llama3.2:1b
- qwen2.5:1.5b

Ready to begin.


In [9]:
## just creating  a reusable function

We will use one function for all our prompts.

It will return:

- The model's answer.
- The time taken.
- The generated-token count reported by Ollama.
- The reason generation stopped.

Reasoning-capable models may use some of the generation allowance for thinking.
An empty or incomplete answer may require a larger allowance.

We will explore reasoning controls separately in Notebook 03.

SyntaxError: unterminated string literal (detected at line 7) (2508366132.py, line 7)

In [10]:
def ask_model(prompt):
    """Send a prompt to Ollama and return the answer and metadata."""

    start = time.perf_counter()

    try:
        response = requests.post(
            f"{OLLAMA_URL}/api/chat",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                "stream": False,
                "options": {
                    "temperature": TEMPERATURE,
                    "num_predict": MAX_GENERATED_TOKENS
                }
            },
            timeout=REQUEST_TIMEOUT
        )

        response.raise_for_status()

    except requests.RequestException as error:
        raise RuntimeError(
            f"Ollama request failed: {error}"
        ) from error

    data = response.json()

    result = {
        "answer": data.get("message", {}).get("content", ""),
        "seconds": round(time.perf_counter() - start, 2),
        "generated_tokens": data.get("eval_count"),
        "stop_reason": data.get("done_reason")
    }

    if not result["answer"].strip():
        print("No final answer was returned. Check the generation allowance.")

    if result["stop_reason"] == "length":
        print("Generation reached its limit. The answer may be incomplete.")

    return result


results = {}


def run_example(label, prompt):
    """Display a prompt and its answer, and store the result."""

    print("PROMPT")
    print("-" * 60)
    print(prompt)

    result = ask_model(prompt)

    results[label] = {
        "prompt": prompt,
        **result
    }

    display(Markdown("### Model response"))
    display(Markdown(result["answer"]))

    print("Time:", result["seconds"], "seconds")
    print("Generated tokens:", result["generated_tokens"])

    return result

In [11]:
## Lets start with a vague prompt

A vague prompt leaves many decisions to the model.

Before running the next cell, consider:

- Which project are we discussing?
- What happened this week?
- Who will read the update?
- How long should it be?
- What format should it use?

Observe whether the model asks questions, creates a generic template, or supplies unsupported details.

SyntaxError: invalid syntax (93131211.py, line 3)

In [12]:
prompt_1 = "Prepare a project update."

result_1 = run_example(
    "1. Vague prompt",
    prompt_1
)

PROMPT
------------------------------------------------------------
Prepare a project update.


### Model response

Sure, I can help you prepare a project update. Here’s a general outline you can follow:

### Project Update Template

**Project Title:** [Insert Project Name Here]

**Date of Update:** [Insert Date Here]

---

### 1. Introduction
- Briefly summarize the project's purpose and main objectives.
- Mention the project's current status and any significant milestones achieved.

### 2. Progress Overview
- **Current Milestones:**
  - List and describe the key achievements and progress made so far.
  - Include any major setbacks or challenges encountered and how they were addressed.

- **Remaining Tasks:**
  - Outline what tasks are still pending and when they need to be completed.
  - Mention any upcoming deadlines or deliverables.

### 3. Challenges and Solutions
- **Challenges Faced:**
  - List the obstacles encountered during the project.
  - Explain the reasons behind these challenges and how they were identified.

- **Solutions Implemented:**
  - Describe the strategies and actions taken to overcome these challenges.
  - Provide examples of how these solutions were implemented and their effectiveness.

### 4. Team Performance
- **Team Contributions:**
  - Highlight the contributions of key team members.
  - Discuss any notable achievements or contributions.

- **Team Dynamics:**
  - Describe the team's communication, collaboration, and overall team dynamics.
  - Mention any team-building activities or initiatives.

### 5. Future Plans
- **Next Steps:**
  - Outline the next steps in the project.
  - Discuss any upcoming deliverables or milestones.

- **Future Challenges:**
  - Identify any potential challenges that may arise in the future.
  - Discuss how these challenges can be mitigated or addressed.

### 6. Conclusion
- Summarize the project's progress and its overall impact.
- Express any final thoughts or recommendations for future improvements.

### 7. Contact Information
- Provide contact details for the project manager or team lead.

---

### Notes:
- Ensure that all information is accurate and up-to-date.
- Use bullet points to make the update clear and easy to read.
- Include any relevant images or charts to visually represent progress or challenges.

Feel free to customize this template to fit your specific project's needs!

Time: 11.46 seconds
Generated tokens: 462


In [13]:
##  Specify the task

A task instruction describes what the model should produce.

Instead of asking for any project update, specify:

- The type of update.
- The project.
- The reporting purpose.

However, a clearer task still needs project facts.

In [20]:
prompt_2 = """
Prepare a weekly status update for the
Customer Support Analytics project.
""".strip()

result_2 = run_example(
    "2. Clear task",
    prompt_2
)

PROMPT
------------------------------------------------------------
Prepare a weekly status update for the
Customer Support Analytics project.


### Model response

**Weekly Status Update for Customer Support Analytics Project**

**Date:** [Insert Date]

---

**1. Project Overview:**
This week, we have made significant progress in refining our customer support analytics project. Our primary focus has been on enhancing the accuracy and efficiency of our data analysis tools. We have also begun implementing new features that will improve user experience and data visualization.

**2. Key Achievements:**

- **Data Cleaning and Integration:** We have completed the initial phase of cleaning and integrating customer support data from various sources. This includes merging data from different systems, resolving inconsistencies, and ensuring data quality.

- **Feature Development:** We have developed a new feature that allows users to filter and search for specific customer support issues more easily. This feature has been tested and is now live, improving the overall user experience.

- **Analytics Enhancements:** We have enhanced our analytics dashboard to provide more detailed insights into customer support trends. This includes adding new metrics and visualizations that help in better understanding customer behavior and support needs.

- **User Feedback:** We have received positive feedback from users regarding the new features. This includes suggestions for further improvements and suggestions for more comprehensive data analysis tools.

**3. Challenges and Solutions:**

- **Data Integration Issues:** We encountered some issues with integrating data from multiple sources. To address this, we have developed a more robust data integration process that includes automated data mapping and validation checks.

- **User Adoption:** There has been a slight resistance to the new features, particularly among users who prefer traditional methods. To address this, we have organized training sessions and provided clear documentation to help users understand the new features.

**4. Next Steps:**

- **Data Analysis:** We will continue to refine our data analysis tools, focusing on improving the accuracy and speed of our algorithms.

- **Feature Expansion:** We plan to expand the new feature that allows for more detailed issue filtering and searching. This will be implemented in the next two weeks.

- **User Training:** We will continue to provide training sessions and documentation to help users adopt the new features more easily.

- **Analytics Enhancements:** We will continue to enhance our analytics dashboard, adding more detailed insights and visualizations.

**5. Conclusion:**
Overall, this week has been productive, with significant progress made in refining our data analysis tools and improving user experience. We are confident that these improvements will lead to better insights and a more efficient customer support process. We will continue to monitor user feedback and make further improvements as needed.

**Contact:**
For any further questions or concerns, please feel free to reach out to [Your Name] at [Your Email] or [Your Phone Number].

---

**End of Weekly Status Update**

Time: 5.29 seconds
Generated tokens: 545


In [ ]:
## 5. Provide context

Context gives the model information it needs to complete the task.

Relevant context might include:

- Project goals.
- Completed work.
- Current blockers.
- Responsibilities.
- Dates and dependencies.

The following facts describe a fictional workshop scenario.

In [21]:
PROJECT_FACTS = """
Project: Customer Support Analytics

Reporting period: 7–11 September 2026

Goal:
Build a dashboard that helps managers identify recurring support issues.

Progress:
- Data cleaning is complete.
- The dashboard prototype is complete.

Blocker:
- User acceptance testing has not started because
  the testing environment is unavailable.

Next actions:
- Priya will restore testing environment access by 14 September 2026.
- Testing is planned to start on 15 September 2026
  if access is restored.

Launch:
- The original launch target is 16 September 2026.
- No revised launch date has been confirmed.

Missing information:
- The testing owner was not provided.
- The launch owner was not provided.
""".strip()


prompt_3 = f"""
Prepare a weekly status update for this project.

Project facts:
{PROJECT_FACTS}
""".strip()

result_3 = run_example(
    "3. Task and context",
    prompt_3
)

PROMPT
------------------------------------------------------------
Prepare a weekly status update for this project.

Project facts:
Project: Customer Support Analytics

Reporting period: 7–11 September 2026

Goal:
Build a dashboard that helps managers identify recurring support issues.

Progress:
- Data cleaning is complete.
- The dashboard prototype is complete.

Blocker:
- User acceptance testing has not started because
  the testing environment is unavailable.

Next actions:
- Priya will restore testing environment access by 14 September 2026.
- Testing is planned to start on 15 September 2026
  if access is restored.

Launch:
- The original launch target is 16 September 2026.
- No revised launch date has been confirmed.

Missing information:
- The testing owner was not provided.
- The launch owner was not provided.


### Model response

**Weekly Status Update for Customer Support Analytics Project**

**Project Facts:**
- Project: Customer Support Analytics
- Reporting Period: 7–11 September 2026
- Goal: Build a dashboard that helps managers identify recurring support issues.

**Progress:**
- Data cleaning is complete.
- The dashboard prototype is complete.

**Blocker:**
- User acceptance testing has not started because the testing environment is unavailable.

**Next Actions:**
- Priya will restore testing environment access by 14 September 2026.
- Testing is planned to start on 15 September 2026 if access is restored.

**Launch:**
- The original launch target is 16 September 2026.
- No revised launch date has been confirmed.

**Missing Information:**
- The testing owner was not provided.
- The launch owner was not provided.

**Summary:**
The project is currently in the data cleaning and prototype development phase. The main blocker is the lack of a testing environment, which has been identified as the reason for the delay. The testing environment is expected to be restored by 14 September 2026, allowing for the start of user acceptance testing on 15 September 2026. The original launch date of 16 September 2026 remains unchanged, but no new launch date has been confirmed. The testing and launch owners have not been assigned, which is a critical oversight that needs to be addressed to ensure the project's success.

Time: 4.06 seconds
Generated tokens: 314


In [ ]:
## Specify the audience

In [ ]:

# The same facts can be communicated differently to different readers.

# Examples:

# - Senior management needs progress, risks, and decisions.
# - A technical team needs dependencies and implementation details.
# - A customer needs clear implications and expectations.

# Audience instructions influence emphasis, terminology, and detail.

In [22]:
prompt_4 = f"""
{prompt_3}

Audience:
Senior management.

Use plain language.
Emphasize completed work, current blockers, and the launch implication.
""".strip()

result_4 = run_example(
    "4. Add audience",
    prompt_4
)

PROMPT
------------------------------------------------------------
Prepare a weekly status update for this project.

Project facts:
Project: Customer Support Analytics

Reporting period: 7–11 September 2026

Goal:
Build a dashboard that helps managers identify recurring support issues.

Progress:
- Data cleaning is complete.
- The dashboard prototype is complete.

Blocker:
- User acceptance testing has not started because
  the testing environment is unavailable.

Next actions:
- Priya will restore testing environment access by 14 September 2026.
- Testing is planned to start on 15 September 2026
  if access is restored.

Launch:
- The original launch target is 16 September 2026.
- No revised launch date has been confirmed.

Missing information:
- The testing owner was not provided.
- The launch owner was not provided.

Audience:
Senior management.

Use plain language.
Emphasize completed work, current blockers, and the launch implication.


### Model response

### Weekly Status Update: Customer Support Analytics Project

**Project Facts:**
- **Project:** Customer Support Analytics
- **Reporting Period:** 7–11 September 2026
- **Goal:** Build a dashboard that helps managers identify recurring support issues.
- **Progress:**
  - Data cleaning is complete.
  - The dashboard prototype is complete.
- **Blocker:**
  - User acceptance testing has not started because the testing environment is unavailable.
- **Next Actions:**
  - Priya will restore testing environment access by 14 September 2026.
  - Testing is planned to start on 15 September 2026 if access is restored.
- **Launch:**
  - The original launch target is 16 September 2026.
  - No revised launch date has been confirmed.
- **Missing Information:**
  - The testing owner was not provided.
  - The launch owner was not provided.

**Audience:**
- Senior management.

### Summary:
The project has made significant progress in data cleaning and dashboard prototype development. However, the primary blocker remains the availability of the testing environment, which is expected to be restored by 14 September 2026. As a result, user acceptance testing is currently not scheduled to begin until 15 September 2026. The original launch date of 16 September 2026 is still the target, but a revised date has not been confirmed. The testing and launch owners are still missing, which could impact the timeline and communication with stakeholders.

This status update highlights the current progress and the remaining challenges, emphasizing the importance of the testing environment and the need for additional information to finalize the launch plan.

Time: 4.35 seconds
Generated tokens: 359


In [ ]:
## 7. Define constraints

In [ ]:
# Constraints specify requirements the answer should satisfy.

# Examples:

# - Maximum word count.
# - Required tone.
# - Information the model may use.
# - How to handle missing details.
# - Claims the model should avoid.

# Constraints should be clear enough that we can check whether they were met.

# A factuality instruction helps guide the model, but the answer still needs review.

In [23]:
prompt_5 = f"""
{prompt_4}

Requirements:
- Use only the supplied project facts.
- Keep the update within 150 words.
- Do not invent percentages, budgets, owners, or dates.
- Mark missing owners or deadlines as "Not provided".
- Describe the launch as at risk.
- Do not claim a confirmed delay or invent a revised launch date.
""".strip()

result_5 = run_example(
    "5. Add constraints",
    prompt_5
)

PROMPT
------------------------------------------------------------
Prepare a weekly status update for this project.

Project facts:
Project: Customer Support Analytics

Reporting period: 7–11 September 2026

Goal:
Build a dashboard that helps managers identify recurring support issues.

Progress:
- Data cleaning is complete.
- The dashboard prototype is complete.

Blocker:
- User acceptance testing has not started because
  the testing environment is unavailable.

Next actions:
- Priya will restore testing environment access by 14 September 2026.
- Testing is planned to start on 15 September 2026
  if access is restored.

Launch:
- The original launch target is 16 September 2026.
- No revised launch date has been confirmed.

Missing information:
- The testing owner was not provided.
- The launch owner was not provided.

Audience:
Senior management.

Use plain language.
Emphasize completed work, current blockers, and the launch implication.

Requirements:
- Use only the supplied projec

### Model response

### Weekly Status Update

**Project: Customer Support Analytics**

**Reporting Period: 7–11 September 2026**

**Goal:**
Build a dashboard to help managers identify recurring support issues.

**Progress:**
- Data cleaning completed.
- Dashboard prototype complete.

**Blocker:**
- Testing environment unavailable, preventing user acceptance testing.

**Next Actions:**
- Priya will restore testing environment access by 14 September 2026.
- Testing is planned to start on 15 September 2026 if access is restored.

**Launch:**
- Original launch target is 16 September 2026.
- No confirmed revised launch date.

**Missing Information:**
- Testing owner not provided.
- Launch owner not provided.

**Audience:**
Senior management.

**Use Plain Language:**
- Completed work: Data cleaning and dashboard prototype.
- Current blockers: Testing environment unavailable.
- Launch implication: At risk due to testing environment issue.

**Requirements:**
- Use only the supplied project facts.
- No percentages, budgets, owners, or dates claimed.
- No inventing deadlines or dates.
- No claimed delay or confirmed revised launch date.

Time: 3.69 seconds
Generated tokens: 247


In [ ]:
## 8. Specify the output format

In [15]:
# An output format makes the answer easier to use.

# Possible formats include:

# - An email.
# - A table.
# - A list of action items.
# - A report with named sections.
# - JSON for further processing.

# Choose the format based on what the reader or application needs.

In [24]:
prompt_6 = f"""
{prompt_5}

Output format:

Title: Customer Support Analytics — Weekly Update

Use these sections:
1. Completed work
2. Blockers and launch implication
3. Next actions

For each next action, include the owner and date when supplied.
Mark missing information as "Not provided".
""".strip()

result_6 = run_example(
    "6. Add output format",
    prompt_6
)

PROMPT
------------------------------------------------------------
Prepare a weekly status update for this project.

Project facts:
Project: Customer Support Analytics

Reporting period: 7–11 September 2026

Goal:
Build a dashboard that helps managers identify recurring support issues.

Progress:
- Data cleaning is complete.
- The dashboard prototype is complete.

Blocker:
- User acceptance testing has not started because
  the testing environment is unavailable.

Next actions:
- Priya will restore testing environment access by 14 September 2026.
- Testing is planned to start on 15 September 2026
  if access is restored.

Launch:
- The original launch target is 16 September 2026.
- No revised launch date has been confirmed.

Missing information:
- The testing owner was not provided.
- The launch owner was not provided.

Audience:
Senior management.

Use plain language.
Emphasize completed work, current blockers, and the launch implication.

Requirements:
- Use only the supplied projec

### Model response

**Customer Support Analytics — Weekly Update**

**Completed Work:**
- Data cleaning completed.
- Dashboard prototype complete.

**Blockers and Launch Implication:**
- Testing environment unavailable, preventing user acceptance testing.
- Testing scheduled to start on 15 September 2026 if testing environment is restored.
- Original launch target is 16 September 2026, but no revised date has been confirmed.

**Next Actions:**
- Priya will restore testing environment access by 14 September 2026.
- Testing is planned to start on 15 September 2026 if access is restored.
- No revised launch date has been confirmed.

**Missing Information:**
- Testing owner not provided.
- Launch owner not provided.

**Audience:**
Senior management.

**Use Plain Language:**
- Emphasize completed work and current blockers.
- Highlight the launch as at risk due to the testing environment issue.

**Requirements:**
- Use only the supplied project facts.
- Keep the update within 150 words.
- Do not invent percentages, budgets, owners, or dates.
- Mark missing owners or deadlines as "Not provided".
- Describe the launch as at risk.

**Output Format:**

Title: Customer Support Analytics — Weekly Update

**Completed Work:**
- Data cleaning completed.
- Dashboard prototype complete.

**Blockers and Launch Implication:**
- Testing environment unavailable, preventing user acceptance testing.
- Testing scheduled to start on 15 September 2026 if testing environment is restored.
- Original launch target is 16 September 2026, but no revised date has been confirmed.

**Next Actions:**
- Priya will restore testing environment access by 14 September 2026.
- Testing is planned to start on 15 September 2026 if access is restored.
- No revised launch date has been confirmed.

**Missing Information:**
- Testing owner not provided.
- Launch owner not provided.

**Audience:**
- Senior management.

**Use Plain Language:**
- Emphasize completed work and current blockers.
- Highlight the launch as at risk due to the testing environment issue.

**Requirements:**
- Use only the supplied project facts.
- Keep the update within 150 words.
- Do not invent percentages, budgets, owners, or dates.
- Mark missing owners or deadlines as "Not provided".
- Describe the launch as at risk.

Time: 5.16 seconds
Generated tokens: 508


In [17]:
## lets compare the results


We have changed the prompt progressively:

Vague instruction → Clear task → Context → Audience → Constraints → Format

Compare the answers for:

- Relevance.
- Factual support.
- Audience suitability.
- Compliance with requirements.
- Usefulness.

The timing measurements are observations, not a controlled performance benchmark.
Model loading, hardware, prompt length, and answer length affect execution time.

In [25]:
comparison = pd.DataFrame([
    {
        "Prompt version": label,
        "Approximate words": len(item["answer"].split()),
        "Time (seconds)": item["seconds"],
        "Generated tokens": item["generated_tokens"],
        "Stop reason": item["stop_reason"]
    }
    for label, item in results.items()
])

display(comparison)

print(
    "Note: The 150-word requirement was introduced "
    "only in prompt versions 5 and 6."
)

,Prompt version,Approximate words,Time (seconds),Generated tokens,Stop reason
0,1. Vague prompt,341,11.46,462,stop
1,2. Clear task,447,5.29,545,stop
2,3. Task and context,219,4.06,314,stop
3,4. Add audience,247,4.35,359,stop
4,5. Add constraints,157,3.69,247,stop
5,6. Add output format,344,5.16,508,stop


Note: The 150-word requirement was introduced only in prompt versions 5 and 6.


## 10. Evaluate answer quality

Score the first and final answers against our desired deliverable.

| Criterion | 0 points | 1 point | 2 points |
|---|---|---|---|
| Task relevance | Misses the task | Partly useful | Useful weekly update |
| Factual support | Major unsupported claims | Minor unsupported claims | All claims supported |
| Audience fit | Unsuitable | Mixed | Clear management focus |
| Requirements | Mostly missed | Partly met | All requirements met |
| Actionability | Actions unclear | Some clarity | Actions and missing details clear |

Maximum score: 10.

The first prompt did not provide all these requirements.
This comparison demonstrates the value of specifying what you need.

### Discussion

1. Which added component helped most?
2. Did the final answer contain unsupported details?
3. Were missing owners identified?
4. Was the launch risk described accurately?
5. Could we shorten the prompt while preserving its usefulness?